## Linear Classifier

In [ ]:
#import LDA

In [ ]:
import jax.numpy as jnp

class LDA:
    """
    Attempt of Linear Discriminant Analysis implementation as ESL (Chapter 4.3).
    Assuming that all classes share the same covariance matrix.
    """

    def __init__(self):
        self.classes = None # Unique class labels
        self.means = None # Class mean vectors μ_k
        self.priors = None # Class-prior probabilities (π_k^ = N_k / N)
        self.cov = None # Shared covariance matrix
        self.cov_inv = None # Inverse of the covariance matrix

    def fit(self, X, y):
        # Unique class labels
        self.classes = jnp.unique(y)
        n_samples, n_features = X.shape
        K = len(self.classes)

        # Estimate class priors π_k
        self.priors = jnp.array([
            jnp.sum(y == c) / n_samples
            for c in self.classes
        ])
        # Estimate class means μ_k
        self.means = jnp.array([
            jnp.mean(X[y == c], axis=0)
            for c in self.classes
        ])
        # Initializing shared covariance matrix
        cov = jnp.zeros((n_features, n_features))
        for i, c in enumerate(self.classes):
            Xc = X[y == c]
            centered = Xc - self.means[i]
            cov += centered.T @ centered
        cov /= (n_samples - K)
        self.cov = cov

        # Inverse covariance matrix Σ^{-1}
        self.cov_inv = jnp.linalg.inv(cov)

        return self

    def delta(self, X, k):
        """
        Discriminant function δ_k(x) for class k.
        """
        mu_k = self.means[k]
        term1 = X @ self.cov_inv @ mu_k
        term2 = - 0.5 * (mu_k @ self.cov_inv @ mu_k)
        term3 = jnp.log(self.priors[k])

        return term1 + term2 + term3

    def decision_rule(self, X):
        """
        Calculate δ_k(x) for each class.
        """
        rules = []
        for k in range(len(self.classes)):
            rules.append(self.delta(X, k))

        return jnp.stack(rules, axis=1)

    def predict(self, X):
        """
        Classify using
            argmax_k δ_k(x)
        """
        scores = self.decision_rule(X)
        idx = jnp.argmax(scores, axis=1)

        return self.classes[idx]

## Logistic Regression Multiclass

In [ ]:
# import Logistic_Regression

In [ ]:
import jax.numpy as jnp
import jax

class LogisticRegression:
    """
    Attempt of Multiclass Logistic Regression implementation as ESL (Chapter 4.4).
    Using K-1 parameter vectors with the Kth class as reference.
    To handle issues when the Hessian matrix is singular, we use the pseudo-inverse.
    """

    def __init__(self, max_iter=100, tol=1e-6):
        self.max_iter = max_iter # Maximum number of iterations
        self.tol = tol # convergence tolerance on parameter change
        self.theta = None # parameter set of vectors k-1
        self.classes = None # class labels
        self.n_features = None # number of features
        self.K = None # number of classes

    def to_list(self, theta):
        """
        Convert theta into a list of K-1 coefficient vectors.
        """
        betas = []
        start = 0
        for _ in range(self.K - 1):
            betas.append(theta[start:start + self.n_features])
            start += self.n_features

        return betas

    def probability(self, X, theta):
        """
        For classes 1,...,K-1:
            p_k(x) = exp(eta_k) / (1 + sum_{l=1}^{K-1} exp(eta_l))
        For the class K:
            p_K(x) = 1 / (1 + sum_{l=1}^{K-1} exp(eta_l))
        """
        betas = self.to_list(theta) # list of K-1 vectors
        # eta_k = X @ beta_k  for k = 1,...,K-1
        eta = jnp.array([jnp.dot(X, beta) for beta in betas]).T  # (n_samples, K-1)
        exp_eta = jnp.exp(eta) # exp(eta_k)

        denom = 1 + jnp.sum(exp_eta, axis=1, keepdims=True)   # denominator

        probs_no_k = exp_eta / denom # probabilities for classes 1..K-1
        prob_k = 1 / denom # probability for class K

        return jnp.concatenate([probs_no_k, prob_k], axis=1)

    def loss(self, theta, X, y):
        """
            l(θ) = Σ_i log p_{g_i}(x_i;θ)
        """
        probs = self.probability(X, theta)
        n = X.shape[0]
        real = probs[jnp.arange(n), y] # For each x_i, pick the probability corresponding to the class y[i]

        return -jnp.mean(jnp.log(real))

    def fit(self, X, y):
        self.classes = jnp.unique(y)
        self.K = len(self.classes)
        self.n_features = X.shape[1]

        # Initialise theta = 0
        theta = jnp.zeros((self.K - 1) * self.n_features)
        # Optimize operations with jax
        loss_op = jax.jit(self.loss)
        grad_op = jax.jit(jax.grad(self.loss))
        hessian_op = jax.jit(jax.hessian(self.loss))
        for i in range(self.max_iter):
            loss_old = loss_op(theta, X, y)
            grad = grad_op(theta, X, y)
            Hessian = hessian_op(theta, X, y)

            # Solve H * delta = g
            # assuming Hessian is invertible
            delta = jnp.linalg.solve(Hessian, grad) # Newton update
            theta_new = theta - delta # (quasi Newton methond)

            # Step‑halving: if the loss doesnot decrease, we reduce the step by half
            step_factor = 1.0
            while step_factor > 1e-10:
                theta_candidate = theta - step_factor * delta
                loss_new = loss_op(theta_candidate, X, y)
                if loss_new < loss_old:
                    theta_new = theta_candidate
                    break
                step_factor *= 0.5

            # Check we reach tolerance convergence
            if jnp.linalg.norm(theta_new - theta) < self.tol:
                theta = theta_new
                break
            theta = theta_new

            if i % 10 == 0: # step of iterations report
                print(f"Iteration {i}, loss: {loss_old:.6f}")
        self.theta = theta

        return self

    def predict_proba(self, X):
        """
        Return class probabilities for samples in X
        """
        return jnp.array(self.probability(X, self.theta))

    def predict(self, X):
        """
        Predict class labels for samples in X
        """
        probs = self.predict_proba(X)
        return jnp.argmax(probs, axis=1)

## Metrics

In [ ]:
def precision_recall_f1(y_real, y_pred):
    classes = jnp.unique(y_real)
    precisions = []
    recalls = []
    f1s = []

    for c in classes:
        tp = jnp.sum((y_pred == c) & (y_real == c))
        fp = jnp.sum((y_pred == c) & (y_real != c))
        fn = jnp.sum((y_pred != c) & (y_real == c))

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

        precisions.append(precision)
        recalls.append(recall)
        f1s.append(f1)

    return precisions, recalls, f1s

## Download Data

### Future Dataset to be used

In [ ]:
import os
os.makedirs('data', exist_ok=True)
%cd data

/content/data


In [ ]:
!curl -O https://os.unil.cloud.switch.ch/fma/fma_metadata.zip
!curl -O https://os.unil.cloud.switch.ch/fma/fma_small.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  341M  100  341M    0     0  25.8M      0  0:00:13  0:00:13 --:--:-- 29.6M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 7323M  100 7323M    0     0  30.4M      0  0:04:00  0:04:00 --:--:-- 30.8M


file verification

In [ ]:
!echo "f0df49ffe5f2a6008d7dc83c6915b31835dfe733  fma_metadata.zip" | sha1sum -c -
#!echo "ade154f733639d52e35e32f5593efe5be76c6d70  fma_small.zip" | sha1sum -c -

fma_metadata.zip: OK
fma_small.zip: OK


In [ ]:
!unzip -q fma_metadata.zip
#!unzip -q fma_small.zip
%cd ..

/content


## Preprocessing data

In [ ]:
import pandas as pd
import numpy as np
import jax.numpy as jnp

raw_data = pd.read_csv('/content/features_3_sec.csv')

raw_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9990 entries, 0 to 9989
Data columns (total 60 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   filename                 9990 non-null   object 
 1   length                   9990 non-null   int64  
 2   chroma_stft_mean         9990 non-null   float64
 3   chroma_stft_var          9990 non-null   float64
 4   rms_mean                 9990 non-null   float64
 5   rms_var                  9990 non-null   float64
 6   spectral_centroid_mean   9990 non-null   float64
 7   spectral_centroid_var    9990 non-null   float64
 8   spectral_bandwidth_mean  9990 non-null   float64
 9   spectral_bandwidth_var   9990 non-null   float64
 10  rolloff_mean             9990 non-null   float64
 11  rolloff_var              9990 non-null   float64
 12  zero_crossing_rate_mean  9990 non-null   float64
 13  zero_crossing_rate_var   9990 non-null   float64
 14  harmony_mean            

In [ ]:
raw_data.head()

,filename,length,chroma_stft_mean,chroma_stft_var,rms_mean,rms_var,spectral_centroid_mean,spectral_centroid_var,spectral_bandwidth_mean,spectral_bandwidth_var,...,mfcc16_var,mfcc17_mean,mfcc17_var,mfcc18_mean,mfcc18_var,mfcc19_mean,mfcc19_var,mfcc20_mean,mfcc20_var,label
0,blues.00000.0.wav,66149,0.335406,0.091048,0.130405,0.003521,1773.065032,167541.630869,1972.744388,117335.771563,...,39.687145,-3.241280,36.488243,0.722209,38.099152,-5.050335,33.618073,-0.243027,43.771767,blues
1,blues.00000.1.wav,66149,0.343065,0.086147,0.112699,0.001450,1816.693777,90525.690866,2010.051501,65671.875673,...,64.748276,-6.055294,40.677654,0.159015,51.264091,-2.837699,97.030830,5.784063,59.943081,blues
2,blues.00000.2.wav,66149,0.346815,0.092243,0.132003,0.004620,1788.539719,111407.437613,2084.565132,75124.921716,...,67.336563,-1.768610,28.348579,2.378768,45.717648,-1.938424,53.050835,2.517375,33.105122,blues
3,blues.00000.3.wav,66149,0.363639,0.086856,0.132565,0.002448,1655.289045,111952.284517,1960.039988,82913.639269,...,47.739452,-3.841155,28.337118,1.218588,34.770935,-3.580352,50.836224,3.630866,32.023678,blues
4,blues.00000.4.wav,66149,0.335579,0.088129,0.143289,0.001701,1630.656199,79667.267654,1948.503884,60204.020268,...,30.336359,0.664582,45.880913,1.689446,51.363583,-3.392489,26.738789,0.536961,29.146694,blues


In [ ]:
target = 'label'

In [ ]:
genres = raw_data[target].unique()

In [ ]:
id_genre = {genre: i for i, genre in enumerate(genres)}
print(id_genre)

{'blues': 0, 'classical': 1, 'country': 2, 'disco': 3, 'hiphop': 4, 'jazz': 5, 'metal': 6, 'pop': 7, 'reggae': 8, 'rock': 9}


In [ ]:
raw_data['label_id'] = raw_data[target].map(id_genre)

In [ ]:
raw_data.head()

,filename,length,chroma_stft_mean,chroma_stft_var,rms_mean,rms_var,spectral_centroid_mean,spectral_centroid_var,spectral_bandwidth_mean,spectral_bandwidth_var,...,mfcc17_mean,mfcc17_var,mfcc18_mean,mfcc18_var,mfcc19_mean,mfcc19_var,mfcc20_mean,mfcc20_var,label,label_id
0,blues.00000.0.wav,66149,0.335406,0.091048,0.130405,0.003521,1773.065032,167541.630869,1972.744388,117335.771563,...,-3.241280,36.488243,0.722209,38.099152,-5.050335,33.618073,-0.243027,43.771767,blues,0
1,blues.00000.1.wav,66149,0.343065,0.086147,0.112699,0.001450,1816.693777,90525.690866,2010.051501,65671.875673,...,-6.055294,40.677654,0.159015,51.264091,-2.837699,97.030830,5.784063,59.943081,blues,0
2,blues.00000.2.wav,66149,0.346815,0.092243,0.132003,0.004620,1788.539719,111407.437613,2084.565132,75124.921716,...,-1.768610,28.348579,2.378768,45.717648,-1.938424,53.050835,2.517375,33.105122,blues,0
3,blues.00000.3.wav,66149,0.363639,0.086856,0.132565,0.002448,1655.289045,111952.284517,1960.039988,82913.639269,...,-3.841155,28.337118,1.218588,34.770935,-3.580352,50.836224,3.630866,32.023678,blues,0
4,blues.00000.4.wav,66149,0.335579,0.088129,0.143289,0.001701,1630.656199,79667.267654,1948.503884,60204.020268,...,0.664582,45.880913,1.689446,51.363583,-3.392489,26.738789,0.536961,29.146694,blues,0


In [ ]:
X = raw_data.drop(['filename', 'length', 'label', 'label_id'], axis=1)
y = raw_data['label_id']

In [ ]:
X.head()

,chroma_stft_mean,chroma_stft_var,rms_mean,rms_var,spectral_centroid_mean,spectral_centroid_var,spectral_bandwidth_mean,spectral_bandwidth_var,rolloff_mean,rolloff_var,...,mfcc16_mean,mfcc16_var,mfcc17_mean,mfcc17_var,mfcc18_mean,mfcc18_var,mfcc19_mean,mfcc19_var,mfcc20_mean,mfcc20_var
0,0.335406,0.091048,0.130405,0.003521,1773.065032,167541.630869,1972.744388,117335.771563,3714.560359,1.080790e+06,...,-2.853603,39.687145,-3.241280,36.488243,0.722209,38.099152,-5.050335,33.618073,-0.243027,43.771767
1,0.343065,0.086147,0.112699,0.001450,1816.693777,90525.690866,2010.051501,65671.875673,3869.682242,6.722448e+05,...,4.074709,64.748276,-6.055294,40.677654,0.159015,51.264091,-2.837699,97.030830,5.784063,59.943081
2,0.346815,0.092243,0.132003,0.004620,1788.539719,111407.437613,2084.565132,75124.921716,3997.639160,7.907127e+05,...,4.806280,67.336563,-1.768610,28.348579,2.378768,45.717648,-1.938424,53.050835,2.517375,33.105122
3,0.363639,0.086856,0.132565,0.002448,1655.289045,111952.284517,1960.039988,82913.639269,3568.300218,9.216524e+05,...,-1.359111,47.739452,-3.841155,28.337118,1.218588,34.770935,-3.580352,50.836224,3.630866,32.023678
4,0.335579,0.088129,0.143289,0.001701,1630.656199,79667.267654,1948.503884,60204.020268,3469.992864,6.102111e+05,...,2.092937,30.336359,0.664582,45.880913,1.689446,51.363583,-3.392489,26.738789,0.536961,29.146694


In [ ]:
y.head()

,label_id
0,0
1,0
2,0
3,0
4,0


## Convert to jax array

In [ ]:
X_jax = jnp.array(X)
y_jax = jnp.array(y)

## Split Data

In [ ]:
def train_test_split(X, y, test_size=0.2, random_state=42):
    np.random.seed(random_state)
    indices = np.random.permutation(len(X))
    test_length = int(len(X) * test_size)
    test_idx = indices[:test_length]
    train_idx = indices[test_length:]
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

X_train, X_test, y_train, y_test = train_test_split(X_jax, y_jax, test_size=0.2)

## Standarize data

In [ ]:
mean = jnp.mean(X_train, axis=0)
std = jnp.std(X_train, axis=0)

# Avoid division by 0 with std = 0
std = jnp.where(std == 0, 1.0, std)

# Standarize
X_train_scaled = (X_train - mean) / std
X_test_scaled = (X_test - mean) / std

## Add beta_0 to X

In [ ]:
X_train_aug = jnp.concatenate([jnp.ones((X_train_scaled.shape[0], 1)), X_train_scaled], axis=1)
X_test_aug = jnp.concatenate([jnp.ones((X_test_scaled.shape[0], 1)), X_test_scaled], axis=1)

## Train models

Linear classification

In [ ]:
lda_model = LDA()
lda_model.fit(X_train_scaled, y_train)
y_pred_lda = lda_model.predict(X_test_scaled)

In [ ]:
print(y_pred_lda)

[8 5 0 ... 3 4 8]


In [ ]:
metrics_lda = precision_recall_f1(y_test, y_pred_lda)
lda_precision = jnp.mean(jnp.array(metrics_lda[0]))
lda_recall = jnp.mean(jnp.array(metrics_lda[1]))
lda_f1 = jnp.mean(jnp.array(metrics_lda[2]))

print('precision: ', lda_precision, ', recall: ', lda_recall, ', f1: ', lda_f1)

precision:  0.6754095 , recall:  0.6771378 , f1:  0.67421395


Logistic Regression

In [ ]:
logistic_model = LogisticRegression()
logistic_model.fit(X_train_aug, y_train)
y_pred_log = logistic_model.predict(X_test_aug)

Iteration 0, loss: 2.302585
Iteration 10, loss: 0.757990
Iteration 20, loss: 0.757854
Iteration 30, loss: 0.757772
Iteration 40, loss: 0.757717
Iteration 50, loss: 0.757681
Iteration 60, loss: 0.757654
Iteration 70, loss: 0.757633
Iteration 80, loss: 0.757617
Iteration 90, loss: 0.757604


In [ ]:
print(y_pred_log)

[0 5 0 ... 0 3 8]


In [ ]:
metrics_log = precision_recall_f1(y_test, y_pred_log)
log_precision = jnp.mean(jnp.array(metrics_log[0]))
log_recall = jnp.mean(jnp.array(metrics_log[1]))
log_f1 = jnp.mean(jnp.array(metrics_log[2]))

print('precision: ', log_precision, ', recall: ', log_recall, ', f1: ', log_f1)

precision:  0.73428255 , recall:  0.73971933 , f1:  0.7354728
